## Load Config 

In [1]:
import sys
from pathlib import Path

import os
from dotenv import load_dotenv
import json
sys.path.insert(0, '..')
# Load variables from .env into the environment
load_dotenv()

# Access the variables
api_key = os.getenv("OPENAI_API_KEY") 
from utils.config_loader import load_config
config = load_config('../config/config.yaml')
print(config)

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from core.state import PipelineState
from agents.watchlist_agent import WatchlistContextAgent
from agents.retrieval_agent import NewsRetrievalAgent
from agents.market_data_agent import MarketDataAgent
from agents.filter_agent import NoiseFilterAgent
from agents.filter_critic_agent import FilterCriticAgent
from agents.clustering_agent import EventClusteringAgent
from agents.summarization_agent import ImpactSummarizationAgent
from agents.ranking_agent import ImportanceRankingAgent
from agents.notification_agent import NotificationAgent

{'retrieval': {'lookback_hours': 720, 'sources': {'sgx_announcements': True, 'newsapi': True, 'yahoo_finance': True, 'cna': True, 'straits_times': True, 'mas': True, 'sbr': True, 'reuters': True, 'nikkei_asia': True, 'cnbc_asia': True, 'reddit': False}, 'max_articles_per_query': 20, 'fetch_full_content': True}, 'newsapi_key': 'b534ae4e571a48d285f602636c3bc4e2', 'newsapi_page_size': 20, 'sgx_request_delay': 1.5, 'sgx_max_pages': 10, 'retrieval_max_workers': 8, 'filtering': {'min_snippet_length': 20, 'similarity_threshold': 0.85, 'llm_model': 'gpt-4o', 'llm_enabled': True}, 'clustering': {'method': 'agglomerative', 'min_cluster_size': 1, 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'}, 'summarization': {'llm_model': 'gpt-4o', 'max_tokens': 512, 'temperature': 0.2, 'verification_enabled': True}, 'ranking': {'weights': {'event_type': 0.4, 'corroboration': 0.25, 'novelty': 0.2, 'credibility': 0.15}, 'source_credibility': {'tier_1': ['SGX Announcements', 'MAS'], 'tier_2': ['Busi

In [2]:
initial = PipelineState()
#config['filtering']['llm_enabled'] = False
watchlist_agent = WatchlistContextAgent(config)
market_data_agent = MarketDataAgent(config)
retrieval_agent = NewsRetrievalAgent(config)
filter_agent = NoiseFilterAgent(config)
filter_critic_agent = FilterCriticAgent(config)
event_clustering_agent = EventClusteringAgent(config)
summarization_agent = ImpactSummarizationAgent(config)
ranking_agent = ImportanceRankingAgent(config)
notification_agent = NotificationAgent(config)

## Agent Watchlist Test

In [3]:
initial.watchlist = ["AAPL", "GOOGL", "AMZN"]
wa_agent_result = watchlist_agent.run(initial)

[2026-04-07 21:52:18] INFO WatchlistContextAgent — [WatchlistContextAgent] Starting. 3 tickers: ['AAPL', 'GOOGL', 'AMZN']
[2026-04-07 21:52:22] INFO WatchlistContextAgent — [WatchlistContextAgent] Done. 3 bundles produced


In [4]:
wa_agent_result.to_json("watchlist_test_output.json")

## Agent Watchlist Load

In [5]:
with open("watchlist_test_output.json", "r") as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
wa_agent_result = PipelineState(**data)

## Retrieval Agent

In [6]:
re_agent_result =  retrieval_agent.run(wa_agent_result) 
re_agent_result

[2026-04-07 21:52:38] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Starting. Fetching from 5 sources for 3 tickers
[2026-04-07 21:52:42] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Done. [Agent 2] ✓ 227 articles (Yahoo Finance: 53, Seeking Alpha: 11, CNBC: 7, Fortune Business Insights: 6, Investor's Business Daily: 4, Investing.com: 4, Barron's: 4, Search Engine Land: 4, Search Engine Journal: 4, Search Engine Roundtable: 4, Watcher Guru: 3, Counterpoint Technology Market Research: 3, 9to5Mac: 3, Benzinga: 2, nationaltoday.com: 2, TipRanks: 2, MarketWatch: 2, The Motley Fool: 2, Business Insider: 2, Quiver Quantitative: 2, Forbes: 2, business.com: 2, TradingView: 2, SQ Magazine: 2, Retail Dive: 2, qz.com: 2, About Amazon: 2, pymnts.com: 2, Omdia: 2, Reuters: 2, AppleInsider: 2, Statista: 2, Straits Research: 2, Meyka: 1, News.az: 1, StockInvest.us: 1, Yahoo Finance UK: 1, Barchart: 1, thestreet.com: 1, Exchange4Media: 1, Digital Commerce 360: 1, Mashable: 1, MacDailyNews: 1, The G

PipelineState(watchlist=['AAPL', 'GOOGL', 'AMZN'], query_bundles=[{'ticker': 'AAPL', 'company_name': 'Apple Inc.', 'aliases': ['Apple', 'Apple Inc'], 'industry': 'Consumer Electronics', 'company_queries': ['Apple earnings Q4 2025', 'Apple iPhone sales', 'Apple stock news'], 'industry_queries': ['smartphone market trends 2025', 'consumer electronics industry news']}, {'ticker': 'GOOGL', 'company_name': 'Alphabet Inc.', 'aliases': ['Google', 'Alphabet Inc'], 'industry': 'Internet Services', 'company_queries': ['Google earnings Q4 2025', 'Google search engine updates', 'Google stock news'], 'industry_queries': ['internet services market trends 2025', 'digital advertising industry news']}, {'ticker': 'AMZN', 'company_name': 'Amazon.com, Inc.', 'aliases': ['Amazon', 'Amazon.com'], 'industry': 'E-commerce', 'company_queries': ['Amazon earnings Q4 2025', 'Amazon Prime subscription growth', 'Amazon stock news'], 'industry_queries': ['e-commerce market trends 2025', 'online retail industry news

In [7]:
re_agent_result.to_json("retrieval_test_output.json")

## Retrieval Agent Load

In [8]:
with open("retrieval_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
re_agent_result = PipelineState(**data)

## Market Retrieval Agent

In [10]:
mr_agent_result =  market_data_agent.run(re_agent_result.watchlist) 
re_agent_result.market_context = mr_agent_result

[2026-04-07 21:53:45] INFO MarketDataAgent — [MarketDataAgent] Starting. Fetching market data for 3 tickers.
[2026-04-07 21:53:48] INFO MarketDataAgent — [MarketDataAgent] Done. Market data fetched for 3/3 tickers. 0 volume spike(s) detected.


In [11]:
re_agent_result.to_json("market_data_test_output.json")

## Load Market Retrieval Agent

In [12]:
with open("market_data_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
re_agent_result = PipelineState(**data)

## News Filtering Agent

In [13]:
filter_agent_result = filter_agent.run(re_agent_result)

[2026-04-07 21:56:32] INFO NoiseFilterAgent — [NoiseFilterAgent] Starting. 227 raw articles
[2026-04-07 21:56:32] INFO NoiseFilterAgent — After hard filters: 113 articles


d:\NUS\NN2\CS5260_financial_news_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-04-07 21:56:38] INFO NoiseFilterAgent — Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11354.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-07 21:56:42] INFO NoiseFilterAgent — After semantic dedup: 107 articles
[2026-04-07 21:56:50] INFO NoiseFilterAgent — After LLM filtering: 29 articles
[2026-04-07 21:56:50] INFO NoiseFilterAgent — [NoiseFilterAgent] Done. [Agent 3] ✓ 29 articles retained after all passes


In [14]:
filter_agent_result.to_json("filter_test_output.json")

## Load News Filter Agent 

In [15]:
with open("filter_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
filter_agent_result = PipelineState(**data)

## Filter Critic Agent

In [16]:
filter_critic_result = filter_critic_agent.run(filter_agent_result)

[2026-04-07 21:57:19] INFO FilterCriticAgent — [FilterCriticAgent] Starting. Evaluating 29 filtered articles (attempt 2/3)
[2026-04-07 21:57:22] INFO FilterCriticAgent — [FilterCriticAgent] Done. RETRY (score 0.505 < 0.55) — 4 issue(s)


In [17]:
filter_critic_result.to_json("filter_critic_test_output.json")

## Load Filter Critic Agent

In [21]:
with open("filter_critic_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
filter_critic_result = PipelineState(**data)

## Clustering Agent

In [22]:
clustering_result = event_clustering_agent.run(filter_critic_result)

[2026-04-07 21:59:13] INFO EventClusteringAgent — [EventClusteringAgent] Starting. 29 clean articles
[2026-04-07 21:59:20] INFO EventClusteringAgent — [EventClusteringAgent] Done. [Agent 4] ✓ Formed 11 event clusters across 3 tickers


In [23]:
clustering_result.to_json("clustring_test_output.json")

## Load Clustering Agent

In [4]:
with open("clustring_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
clustering_result = PipelineState(**data)

## Summarization Agent

In [5]:
Summarization_result = summarization_agent.run(clustering_result)

[2026-04-07 22:10:32] INFO ImpactSummarizationAgent — [ImpactSummarizationAgent] Starting. 11 event clusters
[2026-04-07 22:11:25] INFO ImpactSummarizationAgent — [ImpactSummarizationAgent] Done. [Agent 5] ✓ Generated 11 event cards (2 high-confidence)


In [6]:
Summarization_result.to_json("summarization_test_output.json")

## Load Summarization Result

In [3]:
with open("summarization_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
Summarization_result = PipelineState(**data)

## Ranking Agent

In [4]:
ranking_result = ranking_agent.run(Summarization_result)

[2026-04-07 22:53:33] INFO ImportanceRankingAgent — [ImportanceRankingAgent] Starting. 11 event cards
[2026-04-07 22:53:33] INFO ImportanceRankingAgent — [ImportanceRankingAgent] Done. [Agent 6] ✓ Ranked 11 events — 1 High / 4 Medium / 6 Low (thresholds: High≥0.7, Med≥0.45)


In [5]:
ranking_result.to_json("ranking_test_output.json")

## Load Ranking Agent

In [6]:
with open("ranking_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
ranking_result = PipelineState(**data)

## Notification Agent

In [7]:
notification_result = notification_agent.run(ranking_result)


[2026-04-07 22:53:53] INFO NotificationAgent — [NotificationAgent] Starting. 11 ranked events
[2026-04-07 22:53:53] INFO NotificationAgent — HTML digest saved to output\digest.html
[2026-04-07 22:53:53] INFO NotificationAgent — [NotificationAgent] Done. [Agent 7] ✓ Digest ready — 1 High / 4 Medium / 6 Low


# Load Ranking Agent

In [17]:
from langgraph.graph import StateGraph, END
from core.state import PipelineState

# Import your agent wrappers
from agents.watchlist_agent import watchlist_agent
from agents.retrieval_agent import retrieval_agent
from agents.filter_agent import filter_agent
from agents.filter_critic_agent import filter_critic_agent, route_after_filter_critic

# --- 1. Load the Config Once ---
config = load_config('../config/config.yaml')
#config['filtering']['llm_enabled'] = False

def watchlist_node(state: PipelineState):
    from agents.watchlist_agent import WatchlistContextAgent
    # Combine YAML config with environment keys
    agent = WatchlistContextAgent({**config, "openai_key": os.getenv("OPENAI_API_KEY")})
    return agent.run(state)

def retrieval_node(state: PipelineState):
    from agents.retrieval_agent import NewsRetrievalAgent
    agent = NewsRetrievalAgent(config)
    return agent.run(state)

def filter_node(state: PipelineState):
    from agents.filter_agent import NoiseFilterAgent
    agent = NoiseFilterAgent(config)
    return agent.run(state)

def critic_node(state: PipelineState):
    from agents.filter_critic_agent import FilterCriticAgent
    agent = FilterCriticAgent(config.get("filtering", {}))
    return agent.run(state)

def build_financial_news_pipeline():
    # 1. Initialize the Graph with your Dataclass
    workflow = StateGraph(PipelineState)

    # 2. Add Nodes
    workflow.add_node("resolve_watchlist", watchlist_node)
    workflow.add_node("fetch_news", retrieval_node)
    workflow.add_node("filter_noise", filter_node)
    workflow.add_node("critic_eval", critic_node)
    
    # Placeholder for the next agent in your pipeline (Agent 4)
    # workflow.add_node("cluster_events", event_clustering_agent)

    # 3. Define the Flow (Edges)
    workflow.set_entry_point("resolve_watchlist")
    
    workflow.add_edge("resolve_watchlist", "fetch_news")
    workflow.add_edge("fetch_news", "filter_noise")
    workflow.add_edge("filter_noise", "critic_eval")

    # 4. Add Conditional Routing (The Loop)
    workflow.add_conditional_edges(
        "critic_eval",
        route_after_filter_critic,
        {
            "retry_filter": "filter_noise",  # Loop back to Agent 3
            "continue": END,                # Or link to "cluster_events"
            "abort": END                    # End on critical error
        }
    )

    return workflow.compile()


In [18]:
app = build_financial_news_pipeline()
    
initial_state = PipelineState(watchlist=["TSLA", "NVDA", "AAPL"])
final_state = app.invoke(initial_state)
    
print(f"Final Article Count: {final_state['clean_article_count']}")
print(f"Retries attempted: {final_state['filter_retry_count']}")

[2026-04-07 19:38:22] INFO WatchlistContextAgent — [WatchlistContextAgent] Starting. 3 tickers: ['TSLA', 'NVDA', 'AAPL']
[2026-04-07 19:38:25] INFO WatchlistContextAgent — [WatchlistContextAgent] Done. 3 bundles produced
[2026-04-07 19:38:25] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Starting. Fetching from 5 sources for 3 tickers
[2026-04-07 19:38:26] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Done. [Agent 2] ✓ 234 articles (Yahoo Finance: 58, Seeking Alpha: 13, The Motley Fool: 10, CNBC: 7, Benzinga: 5, Investor's Business Daily: 5, Investing.com: 4, Tom's Hardware: 4, Counterpoint Technology Market Research: 4, Quiver Quantitative: 3, Reuters: 3, Omdia: 3, Automotive News: 2, openpr.com: 2, Autonocion.com: 2, PR Newswire: 2, tradingview.com: 2, Barron's: 2, Autoblog: 2, AP News: 2, TechCrunch: 2, Statista: 2, 9to5Mac: 2, BBC: 2, NVIDIA Newsroom: 2, AppleInsider: 2, Yahoo Finance Singapore: 1, The Driven: 1, thestreet.com: 1, The Economic Times: 1, Binance: 1, EVTech.News: 1

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14306.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-07 19:38:31] INFO NoiseFilterAgent — After semantic dedup: 142 articles
[2026-04-07 19:38:40] INFO NoiseFilterAgent — After LLM filtering: 36 articles
[2026-04-07 19:38:40] INFO NoiseFilterAgent — [NoiseFilterAgent] Done. [Agent 3] ✓ 36 articles retained after all passes
[2026-04-07 19:38:40] INFO FilterCriticAgent — [FilterCriticAgent] Starting. Evaluating 36 filtered articles (attempt 2/3)
[2026-04-07 19:38:43] INFO FilterCriticAgent — [FilterCriticAgent] Done. PASS (score 0.648 ≥ 0.55)
Final Article Count: 36
Retries attempted: 1


In [19]:
final_state['clean_article_count']

36